# `lineitem` — profile, choose columns & trim

Trim type: **row**. Choose **10 columns**, **80,000 rows**. Save `lineitem_10c_80000r.csv`.

> Output columns: `c0`…`c9`.

In [3]:
import os
import numpy as np
import pandas as pd

NAME       = "lineitem"
N_ROWS     = 6001215
N_COLS     = 10
TRIM_ROWS  = "head"   # head | sample

RAW_PATH   = "lineitem.csv"
OUT_PATH   = f"{NAME}_{N_COLS}c_{N_ROWS}r.csv"
DELIM      = ";"
OUT_DELIM  = DELIM
HAS_HEADER = False
ENCODING   = "utf-8"
ON_BAD_LINES = None

## 1. View the raw data

In [4]:
read_kw = dict(sep=DELIM, header=0 if HAS_HEADER else None, encoding=ENCODING, low_memory=False)
if ON_BAD_LINES is not None:
    read_kw["on_bad_lines"] = ON_BAD_LINES
raw = pd.read_csv(RAW_PATH, **read_kw)
raw.columns = [f"c{i}" for i in range(raw.shape[1])]
print("raw shape :", raw.shape)
print("columns   :", list(raw.columns))
raw.head()

raw shape : (6001215, 16)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15']


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12,c13,c14,c15
0,3249924,37439,2446,2,48.0,66068.64,0.04,0.05,R,F,1993-10-05,1993-11-04,1993-11-02,NONE,AIR,ly silent dependenc
1,3249924,23964,1471,3,1.0,1887.96,0.04,0.08,R,F,1993-09-12,1993-10-04,1993-09-28,COLLECT COD,FOB,y. patterns breach.
2,3249925,37271,9775,1,9.0,10874.43,0.10,0.04,N,O,1998-04-19,1998-06-17,1998-04-21,TAKE BACK RETURN,AIR,express instructions among the excuses nag
3,3249925,12918,2919,2,28.0,51265.48,0.07,0.06,N,O,1998-06-18,1998-06-04,1998-06-29,DELIVER IN PERSON,AIR,"eep blithely. silent, even instructions caj"
4,3249925,103351,8372,3,31.0,41984.85,0.02,0.02,N,O,1998-05-12,1998-06-01,1998-06-08,NONE,AIR,ording to the furiously


In [ ]:
raw.dtypes

## 2. Profile: cardinality, top-value %, and group skew

In [5]:
total_rows = len(raw)

rows = []
for col in raw.columns:
    vc = raw[col].value_counts(dropna=False)
    distinct = int(raw[col].nunique(dropna=True))
    nulls    = int(raw[col].isna().sum() + (raw[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,1500000,0,0.0,24.99,0.00,4.00,7,1.75
c1,200000,0,0.0,3.33,0.00,30.01,57,1.90
c2,10000,0,0.0,0.17,0.01,600.12,694,1.16
c3,7,0,0.0,0.00,24.99,857316.43,1500000,1.75
c4,50,0,0.0,0.00,2.01,120024.30,120753,1.01
c5,933900,0,0.0,15.56,0.00,6.43,62,9.65
c6,11,0,0.0,0.00,9.10,545565.00,546395,1.00
c7,9,0,0.0,0.00,11.12,666801.67,667463,1.00
c8,3,0,0.0,0.00,50.72,2000405.00,3043852,1.52


## 3. Choose columns (cardinality mix + id/super-key)

In [6]:
prof_sel = prof[prof.distinct_values > 1]
KEY_THRESH = 95
keys     = prof_sel[prof_sel.unique_percent >= KEY_THRESH]
non_keys = prof_sel[prof_sel.unique_percent <  KEY_THRESH]
chosen = []
if len(keys):
    chosen.append(keys.sort_values("unique_percent", ascending=False).index[0])
n_left = N_COLS - len(chosen)
nk     = non_keys.sort_values("unique_percent")
ix_strata = np.array_split(nk.index.to_numpy(), min(3, max(1, len(nk)))) if len(nk) else []
strata = [nk.loc[ix].sort_values("skew_ratio", ascending=False) for ix in ix_strata]
ptr    = [0] * len(strata)
while n_left > 0 and any(ptr[i] < len(strata[i]) for i in range(len(strata))):
    for i in range(len(strata)):
        if ptr[i] < len(strata[i]):
            chosen.append(strata[i].index[ptr[i]]); ptr[i] += 1; n_left -= 1
            if n_left == 0: break
for c in raw.columns:
    if len(chosen) >= N_COLS: break
    if c not in chosen: chosen.append(c)
SELECTED_COLS = [c for c in raw.columns if c in set(chosen)][:N_COLS]
assert len(SELECTED_COLS) == N_COLS, SELECTED_COLS
print("selected columns:", SELECTED_COLS)

selected columns: ['c1', 'c3', 'c4', 'c5', 'c6', 'c8', 'c10', 'c11', 'c12', 'c15']


## 4. Trim to the chosen columns x exact rows

In [7]:
assert raw.shape[1] >= N_COLS, f"need >= {N_COLS} cols, have {raw.shape[1]}"
n_take = min(N_ROWS, len(raw))
assert len(raw) >= n_take, f"need >= {n_take} rows, have {len(raw)}"

if TRIM_ROWS == "sample":
    trimmed = raw.loc[:, SELECTED_COLS].sample(n_take, random_state=42).reset_index(drop=True)
else:
    trimmed = raw.loc[:, SELECTED_COLS].iloc[:n_take].copy()
trimmed.columns = [f"c{i}" for i in range(N_COLS)]

assert trimmed.shape == (n_take, N_COLS), trimmed.shape
print("trimmed shape:", trimmed.shape)
trimmed.head()

trimmed shape: (6001215, 10)


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9
0,37439,2,48.0,66068.64,0.04,R,1993-10-05,1993-11-04,1993-11-02,ly silent dependenc
1,23964,3,1.0,1887.96,0.04,R,1993-09-12,1993-10-04,1993-09-28,y. patterns breach.
2,37271,1,9.0,10874.43,0.10,N,1998-04-19,1998-06-17,1998-04-21,express instructions among the excuses nag
3,12918,2,28.0,51265.48,0.07,N,1998-06-18,1998-06-04,1998-06-29,"eep blithely. silent, even instructions caj"
4,103351,3,31.0,41984.85,0.02,N,1998-05-12,1998-06-01,1998-06-08,ording to the furiously


## 5. Save the trimmed CSV

In [8]:
out_dir = os.path.dirname(OUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
trimmed.to_csv(OUT_PATH, index=False, sep=OUT_DELIM)
print("wrote", OUT_PATH, trimmed.shape)

chk = pd.read_csv(OUT_PATH, sep=OUT_DELIM)
print("reloaded:", chk.shape)
print("columns   :", list(chk.columns))

wrote lineitem_10c_6001215r.csv (6001215, 10)
reloaded: (6001215, 10)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


## 6. Check selected cardinality and skew

In [9]:
output = pd.read_csv(OUT_PATH, sep=OUT_DELIM, encoding=ENCODING, low_memory=False)
print("output shape :", output.shape)
print("columns      :", list(output.columns))
output.head()

output shape : (6001215, 10)
columns      : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9
0,37439,2,48.0,66068.64,0.04,R,1993-10-05,1993-11-04,1993-11-02,ly silent dependenc
1,23964,3,1.0,1887.96,0.04,R,1993-09-12,1993-10-04,1993-09-28,y. patterns breach.
2,37271,1,9.0,10874.43,0.10,N,1998-04-19,1998-06-17,1998-04-21,express instructions among the excuses nag
3,12918,2,28.0,51265.48,0.07,N,1998-06-18,1998-06-04,1998-06-29,"eep blithely. silent, even instructions caj"
4,103351,3,31.0,41984.85,0.02,N,1998-05-12,1998-06-01,1998-06-08,ording to the furiously


In [10]:
total_rows = len(output)
rows = []
for col in output.columns:
    vc = output[col].value_counts(dropna=False)
    distinct = int(output[col].nunique(dropna=True))
    nulls    = int(output[col].isna().sum() + (output[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))
out_prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")
out_prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,200000,0,0.0,3.33,0.00,30.01,57,1.90
c1,7,0,0.0,0.00,24.99,857316.43,1500000,1.75
c2,50,0,0.0,0.00,2.01,120024.30,120753,1.01
c3,933900,0,0.0,15.56,0.00,6.43,62,9.65
c4,11,0,0.0,0.00,9.10,545565.00,546395,1.00
c5,3,0,0.0,0.00,50.72,2000405.00,3043852,1.52
c6,2526,0,0.0,0.04,0.05,2375.78,2707,1.14
c7,2466,0,0.0,0.04,0.04,2433.58,2667,1.10
c8,2554,0,0.0,0.04,0.04,2349.73,2694,1.15
